In [ ]:
# Cell 1: undetected-chromedriver 설치
# undetected-chromedriver와 selenium을 설치합니다.
%pip install undetected-chromedriver selenium


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.7 MB 4.4 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.7 MB 4.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 4.3 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.7 MB 4.3 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.7 MB 4.3 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 4.4 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 4.3 MB/s eta 0:00:01
   --------

  DEPRECATION: Building 'undetected-chromedriver' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'undetected-chromedriver'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kubernetes 34.1.0 requires urllib3<2.4.0,>=1.24.2, but you have urllib3 2.5.0 which is incompatible.


In [ ]:
# Cell 2: 쿠팡 크롤링 (undetected-chromedriver 사용)
# 봇 감지 우회에 특화된 undetected-chromedriver를 사용합니다.

import sys
import tempfile
import subprocess
import os
import json
import re
from urllib.parse import quote

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요(True로 하면 상품 탐색이 안 됩니다)
NOTEBOOK_HEADLESS = False

# 검색할 키워드 입력 (여기를 수정하세요)
SEARCH_KEYWORD = "축구"  # 예시: "노트북", "무선 이어폰", "스마트폰" 등

# 최대 수집할 상품 개수
MAX_PRODUCTS = 30

# 검색 URL 생성
search_url = f"https://www.coupang.com/np/search?q={quote(SEARCH_KEYWORD)}"

print(f"🔍 검색 키워드: {SEARCH_KEYWORD}")
print(f"🔗 검색 URL: {search_url}")
print(f"📦 최대 수집 개수: {MAX_PRODUCTS}개\n")

# 크롤링 스크립트 생성
crawl_script = f"""import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import json
import time
import re

headless = {NOTEBOOK_HEADLESS!r}
search_url = {search_url!r}
max_products = {MAX_PRODUCTS}

products = []

try:
    # undetected-chromedriver 설정 (봇 감지 우회)
    options = uc.ChromeOptions()
    
    if headless:
        options.add_argument('--headless=new')
    
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--start-maximized')
    
    # User-Agent 설정
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    print("브라우저 시작 중...")
    driver = uc.Chrome(options=options, version_main=None)
    
    print(f"접속 중: {{search_url}}")
    
    # 먼저 쿠팡 메인 페이지로 접속
    print("쿠팡 메인 페이지 접속 중...")
    driver.get("https://www.coupang.com/")
    time.sleep(3)
    
    if not headless:
        time.sleep(10)
    
    # 검색 페이지로 이동
    print(f"\\n검색 페이지로 이동 중: {{search_url}}")
    driver.get(search_url)
    
    # 페이지 로드 대기
    print("페이지 로딩 대기 중...")
    if not headless:
        time.sleep(8)
    else:
        time.sleep(5)
    
    # Access Denied 체크
    page_title = driver.title
    if "Access Denied" in page_title:
        print("⚠ Access Denied 페이지가 감지되었습니다.")
        if not headless:
            print("⚠ 브라우저 창에서 직접 새로고침을 시도해보세요.")
            time.sleep(15)  # 사용자가 처리할 시간 제공
            page_title = driver.title  # 다시 확인
    
    # 스크롤하여 동적 콘텐츠 로드
    print("상품 로딩 중...")
    for scroll_idx in range(4):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.5)")
        time.sleep(3)
    
    time.sleep(3)
    
    # 상품 리스트 찾기
    product_selectors = [
        "li[class*='ProductUnit_productUnit']",
        "li.ProductUnit_productUnit__Qd6sv",
        "ul#product-list li",
        "#productList li",
        "li[data-id]"
    ]
    
    product_items = []
    for selector in product_selectors:
        try:
            items = driver.find_elements(By.CSS_SELECTOR, selector)
            if items and len(items) > 0:
                product_items = items
                print(f"발견된 상품 수: {{len(product_items)}}개 (셀렉터: {{selector}})")
                break
        except Exception as e:
            continue
    
    # 넓은 범위로 재검색
    if not product_items:
        print("\\n기본 셀렉터로 찾지 못함. 넓은 범위로 재검색 중...")
        try:
            all_lis = driver.find_elements(By.TAG_NAME, "li")
            for li in all_lis:
                try:
                    data_id = li.get_attribute("data-id")
                    class_name = li.get_attribute("class") or ""
                    if data_id or "ProductUnit" in class_name or "product" in class_name.lower():
                        product_items.append(li)
                except:
                    continue
            if product_items:
                print(f"넓은 범위 검색으로 {{len(product_items)}}개 요소 발견")
        except Exception as e:
            print(f"넓은 범위 검색 중 오류: {{e}}")
    
    if not product_items:
        print("⚠ 상품 리스트를 찾을 수 없습니다.")
        print(f"현재 URL: {{driver.current_url}}")
        print(f"페이지 제목: {{driver.title}}")
    else:
        # 각 상품 정보 추출
        for idx, item_elem in enumerate(product_items[:max_products]):
            try:
                if idx > 0:
                    time.sleep(0.5)
                
                # 상품명 추출
                title = ""
                title_selectors = [
                    "a.ProductUnit_productName__P8nrl",
                    "div.ProductUnit_productInfo__1l0il a",
                    "div.ProductUnit_productInfo__1l0il",
                    ".name",
                    "[class*='name']"
                ]
                for sel in title_selectors:
                    try:
                        title_elem = item_elem.find_element(By.CSS_SELECTOR, sel)
                        title = title_elem.text.strip()
                        if title:
                            break
                    except:
                        continue
                
                # 상품명 정제 (불필요한 정보 제거)
                if title:
                    # 줄바꿈으로 분리
                    lines = title.split("\\n")
                    cleaned_lines = []
                    
                    for line in lines:
                        line = line.strip()
                        if not line:
                            continue
                        
                        # 가격 패턴 제거 (숫자,원 포함)
                        if re.search(r'[\\d,]+\\s*원', line):
                            continue
                        
                        # 배송 관련 키워드 제거
                        if any(keyword in line for keyword in ["도착", "배송", "무료배송", "로켓배송", "내일", "오늘"]):
                            continue
                        
                        # 리뷰/평점 관련 제거
                        if re.search(r'[\\d.]+\\s*\\([\\d,]+\\)', line) or "리뷰" in line or "평점" in line:
                            continue
                        
                        # 쿠폰/할인 관련 제거
                        if any(keyword in line for keyword in ["쿠폰할인", "할인", "%", "적립", "캐시"]):
                            continue
                        
                        # 기타 불필요한 키워드 제거
                        if any(keyword in line for keyword in ["AD", "새 상품", "반품", "품절", "와우"]):
                            continue
                        
                        cleaned_lines.append(line)
                    
                    # 첫 번째 줄만 사용 (보통 상품명)
                    if cleaned_lines:
                        title = cleaned_lines[0]
                    else:
                        # 정제 후 비어있으면 원본의 첫 번째 줄 사용
                        if lines:
                            title = lines[0].strip()
                
                # 상품 링크 추출
                product_link = ""
                try:
                    link_elem = item_elem.find_element(By.CSS_SELECTOR, "a")
                    href = link_elem.get_attribute("href") or ""
                    if href:
                        if href.startswith("http"):
                            product_link = href
                        elif href.startswith("/"):
                            product_link = f"https://www.coupang.com{{href}}"
                except:
                    pass
                
                # 가격 정보 추출 (텍스트 크기와 스타일로 구분)
                original_price = ""
                displayed_price = ""
                
                # custom-oos 클래스를 가진 모든 요소 찾기
                try:
                    all_custom_oos = item_elem.find_elements(By.CSS_SELECTOR, "[class*='custom-oos']")
                    
                    for elem in all_custom_oos:
                        class_attr = elem.get_attribute("class") or ""
                        tag_name = elem.tag_name.lower()
                        price_text = elem.text.strip()
                        
                        if not price_text or "원" not in price_text:
                            continue
                        
                        # 가격 숫자 추출
                        price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                        if not price_match:
                            price_match = re.search(r'([\\d,]+)', price_text.replace(' ', ''))
                        if not price_match:
                            continue
                        
                        # 가격 값 추출 (쉼표 포함)
                        price_value_raw = price_match.group(1)
                        price_value = price_value_raw.replace(",", "")  # 숫자 비교용
                        
                        # 숫자가 실제로 있는지 확인 (쉼표만 있으면 제외)
                        if not price_value or not price_value.isdigit():
                            continue
                        
                        price_value_formatted = price_value_raw + "원"  # 원화 형식
                        
                        # 원가 판별: del 태그이거나 취소선이 있거나 작은 텍스트 크기
                        is_original = False
                        if tag_name == "del" or "fw-line-through" in class_attr:
                            is_original = True
                        elif "fw-text-[12px]" in class_attr or "fw-text-[14px]" in class_attr:
                            # 작은 텍스트 크기 (12px, 14px)는 원가
                            is_original = True
                        
                        # 원가 저장 (원가로 확실히 판별된 경우만)
                        if is_original and not original_price:
                            original_price = price_value_formatted
                            break  # 첫 번째 원가만 저장
                    
                    # 2단계: 할인가 추출 (원가가 아니고, 큰 텍스트이거나 볼드인 경우만)
                    for elem in all_custom_oos:
                        class_attr = elem.get_attribute("class") or ""
                        tag_name = elem.tag_name.lower()
                        price_text = elem.text.strip()
                        
                        if not price_text or "원" not in price_text:
                            continue
                        
                        # 가격 숫자 추출
                        price_match = re.search(r'([\\\\d,]+)\\\\s*원', price_text)
                        if not price_match:
                            price_match = re.search(r'([\\\\d,]+)', price_text.replace(' ', ''))
                        if not price_match:
                            continue
                        
                        # 가격 값 추출 (쉼표 포함)
                        price_value_raw = price_match.group(1)
                        price_value = price_value_raw.replace(",", "")  # 숫자 비교용
                        
                        # 숫자가 실제로 있는지 확인 (쉼표만 있으면 제외)
                        if not price_value or not price_value.isdigit():
                            continue
                        
                        price_value_formatted = price_value_raw + "원"  # 원화 형식
                        
                        # 이미 원가로 저장된 가격이면 스킵 (숫자만 비교)
                        if original_price:
                            original_price_num = original_price.replace(",", "").replace("원", "")
                            if price_value == original_price_num:
                                continue
                        
                        # 원가 조건 체크 (del 태그, 취소선, 작은 텍스트는 제외)
                        is_original = False
                        if tag_name == "del":
                            is_original = True
                        elif "fw-line-through" in class_attr:
                            is_original = True
                        elif "fw-text-[12px]" in class_attr or "fw-text-[14px]" in class_attr:
                            is_original = True
                        
                        # 원가가 아니고, 할인가 조건을 만족하는 경우만
                        if not is_original:
                            is_discount = False
                            # 큰 텍스트 크기 (20px, 24px)는 할인가
                            if "fw-text-[20px]" in class_attr or "fw-text-[24px]" in class_attr:
                                is_discount = True
                            # 볼드이고 취소선이 없으면 할인가
                            elif "fw-font-bold" in class_attr:
                                is_discount = True
                            
                            # 할인가 저장
                            if is_discount and not displayed_price:
                                displayed_price = price_value_formatted
                                break  # 첫 번째 할인가만 저장
                    
                    # 할인가를 찾지 못했지만 원가는 있는 경우, 큰 텍스트 크기를 가진 요소 재검색
                    if original_price and not displayed_price:
                        for elem in all_custom_oos:
                            class_attr = elem.get_attribute("class") or ""
                            tag_name = elem.tag_name.lower()
                            
                            if tag_name == "del" or "fw-line-through" in class_attr:
                                continue
                            
                            # 큰 텍스트 크기나 볼드인 요소 찾기
                            if ("fw-text-[20px]" in class_attr or 
                                "fw-text-[24px]" in class_attr or 
                                "fw-font-bold" in class_attr):
                                price_text = elem.text.strip()
                                if price_text and "원" in price_text:
                                    price_match = re.search(r'([\\d,]+)\\s*원', price_text)
                                    if not price_match:
                                        price_match = re.search(r'([\\d,]+)', price_text.replace(' ', ''))
                                    if price_match:
                                        price_value_raw = price_match.group(1)
                                        price_value = price_value_raw.replace(",", "")
                                        
                                        # 숫자가 실제로 있는지 확인
                                        if not price_value or not price_value.isdigit():
                                            continue
                                        
                                        # 원가와 다른 가격만 할인가로 저장
                                        original_price_num = original_price.replace(",", "").replace("원", "")
                                        if price_value != original_price_num:
                                            displayed_price = price_value_raw + "원"
                                            break
                    
                    # 여전히 둘 다 비어 있으면 custom-oos에서 첫 가격을 displayed_price로 사용
                    if not original_price and not displayed_price:
                        for elem in all_custom_oos:
                            price_text = elem.text.strip()
                            if not price_text or "원" not in price_text:
                                continue
                            price_match = re.search(r'([\d,]+)\s*원', price_text)
                            if not price_match:
                                price_match = re.search(r'([\d,]+)', price_text.replace(' ', ''))
                            if price_match:
                                price_value_raw = price_match.group(1)
                                price_value = price_value_raw.replace(",", "")
                                
                                # 숫자가 실제로 있는지 확인
                                if not price_value or not price_value.isdigit():
                                    continue
                                
                                displayed_price = price_value_raw + "원"
                                break
                
                except Exception as e:
                    pass
                
                # 썸네일 이미지 URL 추출
                thumbnail_url = ""
                try:
                    img_elem = item_elem.find_element(By.CSS_SELECTOR, "img")
                    thumbnail_url = img_elem.get_attribute("src") or ""
                    if not thumbnail_url:
                        thumbnail_url = img_elem.get_attribute("data-src") or ""
                except:
                    pass
                
                # 최소한 제목이 있어야 유효한 상품
                if title:
                    product_data = {{
                        "title": title,
                        "original_price": original_price,  # 원가 (취소선 있는 가격)
                        "displayed_price": displayed_price,  # 할인된 가격
                        "product_link": product_link,
                        "thumbnail_url": thumbnail_url
                    }}
                    products.append(product_data)
                    
                    if len(products) >= max_products:
                        break
                    
            except Exception as e:
                print(f"  상품 {{idx+1}} 정보 추출 오류: {{e}}")
                continue
    
except Exception as e:
    print(f"⚠ 크롤링 오류: {{e}}")
    import traceback
    traceback.print_exc()
finally:
    try:
        driver.quit()
    except:
        pass

# 결과 저장
result = {{
    "search_keyword": {SEARCH_KEYWORD!r},
    "search_url": search_url,
    "total_products": len(products),
    "products": products
}}

# JSON 파일로 저장
output_json = "coupang_search_results.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"\\n✅ 총 {{len(products)}}개 상품 정보 수집 완료")
print(f"✅ JSON 파일 저장 완료: {{output_json}}")

# 결과 미리보기
print(f"\\n📋 수집된 상품 미리보기 (처음 5개):")
for idx, product in enumerate(products[:5], 1):
    print(f"\\n  {{idx}}. {{product['title'][:60]}}...")
    if product.get('original_price'):
        print(f"     원가: {{product['original_price']}}")
    if product.get('displayed_price'):
        print(f"     할인가격: {{product['displayed_price']}}")
    print(f"     링크: {{product['product_link'][:70]}}..." if product['product_link'] else "     링크: 정보 없음")
"""

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_coupang_uc.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

print("▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...\n")

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


🔍 검색 키워드: 축구
🔗 검색 URL: https://www.coupang.com/np/search?q=%EC%B6%95%EA%B5%AC
📦 최대 수집 개수: 30개

▶ 쿠팡 상품 검색 크롤링을 시작합니다 (undetected-chromedriver 사용)...

브라우저 시작 중...
접속 중: https://www.coupang.com/np/search?q=%EC%B6%95%EA%B5%AC
쿠팡 메인 페이지 접속 중...

검색 페이지로 이동 중: https://www.coupang.com/np/search?q=%EC%B6%95%EA%B5%AC
페이지 로딩 대기 중...
상품 로딩 중...
발견된 상품 수: 35개 (셀렉터: li[class*='ProductUnit_productUnit'])

✅ 총 30개 상품 정보 수집 완료
✅ JSON 파일 저장 완료: coupang_search_results.json

📋 수집된 상품 미리보기 (처음 5개):

  1. 파리 생제르망 축구공 SC-06 투센 볼가방 투센 볼펌프, 1세트...
     원가: 35,000원원
     할인가격: 18,900원원
     링크: https://www.coupang.com/vp/products/8020387941?itemId=22410861446&vend...

  2. 새로이 축구 넥워머 비니 장갑 세트 1+1+1 챔피언스리그 겨울 방한 풋살...
     원가: 53,780원원
     할인가격: 19,800원원
     링크: https://www.coupang.com/vp/products/7822521561?itemId=21250915457&vend...

  3. BEST AWARDS...
     원가: 16,400원원
     할인가격: 9,800원원
     링크: https://www.coupang.com/vp/products/7014369123?itemId=17250350040&vend...

  4. 리츠제이 스포츠 런닝 긴팔 트레이닝복 세트 사계절 